In [17]:
import pandas as pd 
import numpy as np 
import re
from collections import Counter

import scipy.cluster.hierarchy as spc
import matplotlib.pyplot as plt 

In [18]:
def ngram(df, min_shp = 0.1, do_abs=False):
    fn_abs = lambda x: np.abs(x) if do_abs else x
    df_corr = df.corr()
    
    df_corr_sorted = df_corr.copy()
    sep = ',| |"|\\(|\\)|\\+|\\-|/|\\*|\n'

    tokens = re.split(sep, ",".join(df_corr_sorted.index))
    # print(tokens)

    df_freq = pd.Series(Counter(tokens))
    df_freq = df_freq.sort_values(ascending=False)

    co_occurence_hq = pd.DataFrame(0, index=df_freq.index, columns=df_freq.index)
    shp = (df.mean(axis=0) / (df.std(axis=0)))
    eqns_hq= []

    for eqn in df_corr_sorted.index: 
        if not min_shp < (shp[eqn])< 999: 
            continue
        eqns_hq.append(eqn)
        tokens = re.split(sep, eqn)
        for tok1 in tokens:
            for tok2 in tokens:
                co_occurence_hq.loc[tok1,tok2] += 1

    tokens_hq = re.split(sep, ",".join(eqns_hq))
    df_freq_hq = pd.Series(Counter(tokens_hq))
    df_freq_hq = df_freq_hq.sort_values(ascending=False)

    # import numpy as np 
    co_occurence_hq = co_occurence_hq / np.sqrt(np.diag(co_occurence_hq)[np.newaxis,:] * np.diag(co_occurence_hq)[:,np.newaxis])

    # accounts = [x.replace("ifndq_","").replace("afnd1_","") for x in co_occurence_hq.index if (x.startswith("ifndq_") or x.startswith("afnd1"))]
    accounts = ["revtq", "piq", "niq", "xintq", "dlcq", "seqq", "dlttq", "cogsq", "rectq", "dvpq", "atq", "lctq", "invtq", "dvc", "cheq", "apq", "xrent", "txtq", "oiadpq", "oancf", "ceqq", "capx", "actq", "xsgaq", "ppentq", "gpq", "dpq", "cshpri", "cshoq",]
    co_occurence_hq.index = [x.replace("ifndq_","").replace("afnd1_","").lower() for x in co_occurence_hq.index]
    co_occurence_hq.columns = co_occurence_hq.index
    df_freq_hq.index = [x.replace("ifndq_","").replace("afnd1_","").lower() for x in df_freq_hq.index]

    # print(df_freq_hq)
    a = df_freq_hq.reindex(accounts).fillna(0)
    b = co_occurence_hq.reindex(index=accounts, columns=accounts).fillna(0)
    return  a, b


In [19]:
# def load_cfa(path="../wrds/financial_ratios.txt"): // factors without simple inversion or duplicate field. yields weaker result.
def load_cfa(path="../wrds/financial_ratios_nodup.txt"):
    with open(path, "rt") as f:
        # lines = [ [tok.lower() for tok in re.split(',| |"|\\(|\\)|\n', l)] for l in f.readlines() if ("Quarterly" in l)] 
        lines = [l.lower() for l in f.readlines() if ("Quarterly" in l)] 
    return pd.DataFrame([[0]*len(lines),[1]*len(lines)], index=[0,1], columns = lines)
# load_cfa()

### Token frequency correlation, Mined alpha pool vs. Known factors 

In [20]:
df_freq_cfa, co_occurence_cfa = ngram(load_cfa(), min_shp = -1)


In [21]:

df = pd.read_csv("./returns_30.csv", index_col=[0], header=None).T
df_freq_hq, co_occurence_hq = ngram(df, min_shp = 0.0)
pd.DataFrame({"mined": df_freq_hq, "known": df_freq_cfa}).corr(method="spearman")



,mined,known
mined,1.000000,0.113039
known,0.113039,1.000000


In [22]:

df = pd.read_csv("./returns_30.csv", index_col=[0], header=None).T
df_freq_hq, co_occurence_hq = ngram(df, min_shp = 0.1)
pd.DataFrame({"mined": df_freq_hq, "known": df_freq_cfa}).corr(method="spearman")



,mined,known
mined,1.000000,0.103957
known,0.103957,1.000000


In [23]:

df = pd.read_csv("./returns_30.csv", index_col=[0], header=None).T
df_freq_hq, co_occurence_hq = ngram(df, min_shp =-999)
print(len(df_freq_hq))
pd.DataFrame({"mined": df_freq_hq, "known": df_freq_cfa}).corr(method="spearman")

29


,mined,known
mined,1.000000,0.183273
known,0.183273,1.000000


### Two-gram correlation, Mined alpha pool vs. Known factors 

In [24]:
df_hq_revt = co_occurence_hq.stack()[["revtq", "gpq", "cogsq"]].groupby(level=1).mean()

df_hq_revt_transition = pd.Series({
    "sales": df_hq_revt[["revtq", "gpq", "cogsq"]].mean(),
    "assets" : df_hq_revt[["atq", "actq", "cheq", "invtq", "rectq", "ppentq"]].mean(), 
    "liabilities" : df_hq_revt[[ "lctq", "dlcq", "dlttq", "apq"]].mean(),
})
df_hq_revt_transition

sales          0.414250
assets         0.094029
liabilities    0.098477
dtype: float64

In [25]:
df_cfa_revt = co_occurence_cfa.stack()[["revtq", "gpq", "cogsq"]].groupby(level=1).mean()

df_cfa_revt_transition = pd.Series({
    "sales": df_cfa_revt[["revtq", "gpq", "cogsq"]].mean(),
    "assets" : df_cfa_revt[["atq", "actq", "cheq", "invtq", "rectq", "ppentq"]].mean(), 
    "liabilities" : df_cfa_revt[[ "lctq", "dlcq", "dlttq", "apq"]].mean(),
})
df_cfa_revt_transition

sales          0.403606
assets         0.137022
liabilities    0.013176
dtype: float64

In [26]:
df_hq_ast = co_occurence_hq.stack()[["atq", "actq", "cheq", "invtq", "rectq", "ppentq"]].groupby(level=1).mean()

df_hq_ast_transition = pd.Series({
    "sales": df_hq_ast[["revtq", "gpq", "cogsq"]].mean(),
    "assets" : df_hq_ast[["atq", "actq", "cheq", "invtq", "rectq", "ppentq"]].mean(), 
    "liabilities" : df_hq_ast[[ "lctq", "dlcq", "dlttq", "apq"]].mean(),
})
df_hq_ast_transition

sales          0.094029
assets         0.236154
liabilities    0.119419
dtype: float64

In [27]:
df_cfa_ast = co_occurence_cfa.stack()[["atq", "actq", "cheq", "invtq", "rectq", "ppentq"]].groupby(level=1).mean()

df_cfa_ast_transition = pd.Series({
    "sales": df_cfa_ast[["revtq", "gpq", "cogsq"]].mean(),
    "assets" : df_cfa_ast[["atq", "actq", "cheq", "invtq", "rectq", "ppentq"]].mean(), 
    "liabilities" : df_cfa_ast[[ "lctq", "dlcq", "dlttq", "apq"]].mean(),
})
df_cfa_ast_transition

sales          0.137022
assets         0.198742
liabilities    0.075721
dtype: float64